# Trabajo Práctico N°2 — Predicción de Fumadores

**Alumno:** Gonzalo Zarazaga

---

## Dataset: Body Signal of Smoking

### Descripción
El dataset contiene indicadores biomédicos relevados en chequeos médicos de pacientes. El objetivo es predecir si una persona es fumadora o no (`smoking`) a partir de sus señales corporales, incluyendo medidas físicas, análisis de sangre, y examen odontológico.

### Diccionario de datos

| Variable | Descripción |
|---|---|
| `ID` | Identificador único del registro |
| `gender` | Sexo del paciente (F / M) |
| `age` | Edad en años |
| `height(cm)` | Altura en centímetros |
| `weight(kg)` | Peso en kilogramos |
| `waist(cm)` | Circunferencia de cintura en centímetros |
| `eyesight(left)` | Agudeza visual ojo izquierdo |
| `eyesight(right)` | Agudeza visual ojo derecho |
| `hearing(left)` | Audición oído izquierdo (1=normal, 2=anormal) |
| `hearing(right)` | Audición oído derecho (1=normal, 2=anormal) |
| `systolic` | Presión arterial sistólica (mmHg) |
| `relaxation` | Presión arterial diastólica (mmHg) |
| `fasting blood sugar` | Glucosa en sangre en ayunas (mg/dL) |
| `Cholesterol` | Colesterol total (mg/dL) |
| `triglyceride` | Triglicéridos (mg/dL) |
| `HDL` | Colesterol HDL (mg/dL) |
| `LDL` | Colesterol LDL (mg/dL) |
| `hemoglobin` | Hemoglobina en sangre (g/dL) |
| `Urine protein` | Proteína en orina (escala 1–6) |
| `serum creatinine` | Creatinina sérica (mg/dL) |
| `AST` | Enzima hepática AST (U/L) |
| `ALT` | Enzima hepática ALT (U/L) |
| `Gtp` | Gamma-glutamil transferasa (U/L) |
| `oral` | Examen oral (Y=realizado, N=no realizado) |
| `dental caries` | Presencia de caries dentales (0=no, 1=sí) |
| `tartar` | Presencia de sarro (Y=sí, N=no) |
| `smoking` | **Variable objetivo**: fumador (1) o no fumador (0) |

## 1. Lectura del dataset

Se carga el archivo CSV original desde la carpeta `data/raw/` sin ninguna transformación previa.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/raw/smoking_prediction.csv")
df.shape

## 2. Exploración inicial del dataset

Inspección general de la estructura, tipos de datos, rangos y valores únicos antes de cualquier transformación.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.nunique()

In [ ]:
columnas_categoricas = df.select_dtypes(include="object").columns
for col in columnas_categoricas:
    print(f"{col}: {df[col].unique()}")

In [ ]:
df.describe()

In [ ]:
df["smoking"].value_counts()

## 3. Tipos de variables

| Variable | Tipo | Notas |
|---|---|---|
| `ID` | Identificador | Se excluye del modelado |
| `gender` | Categórica nominal | F / M |
| `oral` | Categórica nominal | Y / N |
| `tartar` | Categórica nominal | Y / N |
| `age` | Numérica discreta | En intervalos de 5 años |
| `hearing(left)` | Numérica discreta | 1=normal, 2=anormal |
| `hearing(right)` | Numérica discreta | 1=normal, 2=anormal |
| `dental caries` | Numérica discreta | Binaria: 0 / 1 |
| `Urine protein` | Numérica discreta | Escala 1–6 |
| `height(cm)` | Numérica continua | Altura en cm |
| `weight(kg)` | Numérica continua | Peso en kg |
| `waist(cm)` | Numérica continua | Circunferencia de cintura |
| `eyesight(left)` | Numérica continua | Agudeza visual |
| `eyesight(right)` | Numérica continua | Agudeza visual |
| `systolic` | Numérica continua | Presión sistólica mmHg |
| `relaxation` | Numérica continua | Presión diastólica mmHg |
| `fasting blood sugar` | Numérica continua | Glucosa mg/dL |
| `Cholesterol` | Numérica continua | Colesterol total mg/dL |
| `triglyceride` | Numérica continua | Triglicéridos mg/dL |
| `HDL` | Numérica continua | Colesterol bueno mg/dL |
| `LDL` | Numérica continua | Colesterol malo mg/dL |
| `hemoglobin` | Numérica continua | Hemoglobina g/dL |
| `serum creatinine` | Numérica continua | Creatinina mg/dL |
| `AST` | Numérica continua | Enzima hepática U/L |
| `ALT` | Numérica continua | Enzima hepática U/L |
| `Gtp` | Numérica continua | Gamma-GT U/L |
| `smoking` | **Categórica binaria** | Variable objetivo: 0=no fumador, 1=fumador |

## 4. Análisis de valores nulos

In [ ]:
nulos = df.isna().sum()
print("Valores nulos por columna:")
print(nulos[nulos > 0] if nulos.sum() > 0 else "Sin valores nulos")
print(f"\nTotal de filas con al menos un nulo: {df.isna().any(axis=1).sum()}")

## 5. Detección de valores atípicos

Se utiliza el método **IQR (Rango Intercuartílico)** sobre las variables numéricas continuas.
Un valor se considera atípico si cae fuera del rango:

```
límite inferior = Q1 - 1.5 × IQR
límite superior = Q3 + 1.5 × IQR
```

In [ ]:
columnas_numericas = df.select_dtypes(include="number").columns.drop(["ID", "smoking"])

resumen_outliers = []

for col in columnas_numericas:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR

    outliers = df[(df[col] < limite_inf) | (df[col] > limite_sup)]

    resumen_outliers.append({
        "Variable": col,
        "Q1": round(Q1, 2),
        "Q3": round(Q3, 2),
        "IQR": round(IQR, 2),
        "Límite inferior": round(limite_inf, 2),
        "Límite superior": round(limite_sup, 2),
        "Cantidad de outliers": len(outliers),
        "% del total": round(len(outliers) / len(df) * 100, 2)
    })

df_outliers = pd.DataFrame(resumen_outliers)
df_outliers